# Generator

## 1. Konfiguracja

In [72]:
N_SPK = 2
FS = 48000
GOLAY_N = 17
MLS_M = 17
GOLD_M = 16
GUARD = 96000
GAP_MS = 200.0
SYNC_MS = 40.0
NOISE_FLOOR_MS = 5000.0
THD_FREQS = [100, 250, 500, 1000, 2000, 4000, 8000]
THD_DUR_MS = 2000.0
PEAK = 0.85
OUT_DIR = "bundle_colab"

{"glosniki": N_SPK, "guard_ms": GUARD / FS * 1000}

{'glosniki': 2, 'guard_ms': 2000.0}

In [73]:
import json
import os
from collections import Counter
import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt

## 2. Sekwencje

In [74]:
def golay_pair(n):
    A, B = np.array([1.0]), np.array([1.0])
    for _ in range(n):
        A, B = np.concatenate([A, B]), np.concatenate([A, -B])
    return A, B


MLS_TAPS = {8: [8, 6, 5, 4], 9: [9, 5], 10: [10, 7], 12: [12, 6, 4, 1],
            14: [14, 5, 3, 1], 16: [16, 14, 13, 11], 17: [17, 14], 18: [18, 11]}
GOLD_PAIRS = {10: ([10, 7], [10, 9, 8, 6]), 12: ([12, 6, 4, 1], [12, 11, 10, 4]),
              16: ([16, 14, 13, 11], [16, 15, 13, 4])}


def generate_mls(taps, m):
    reg = [1] * m
    seq = []
    for _ in range(2**m - 1):
        seq.append(1.0 if reg[-1] else -1.0)
        fb = 0
        for t in taps:
            fb ^= reg[t - 1]
        reg = [fb] + reg[:-1]
    return np.array(seq)


def reciprocal_mls(seq):
    return np.concatenate([seq[:1], seq[:0:-1]])


def gold_family(m, n_codes):
    t1, t2 = GOLD_PAIRS[m]
    mls1, mls2 = generate_mls(t1, m), generate_mls(t2, m)
    codes = [mls1, mls2]
    for k in range(len(mls1)):
        if len(codes) >= n_codes:
            break
        codes.append(mls1 * np.roll(mls2, k))
    return codes[:n_codes]


def hadamard_matrix(n):
    H = np.array([[1.0]])
    while H.shape[0] < n:
        H = np.block([[H, H], [H, -H]])
    return H


A, B = golay_pair(GOLAY_N)
mls = generate_mls(MLS_TAPS[MLS_M], MLS_M)
mls_rec = reciprocal_mls(mls)
gold_codes = gold_family(GOLD_M, N_SPK)
H_SIZE = int(2 ** np.ceil(np.log2(N_SPK)))
H = hadamard_matrix(H_SIZE)
L_GOLAY, L_MLS, L_GOLD = A.size, mls.size, gold_codes[0].size

{"golay_L": L_GOLAY, "mls_L": L_MLS, "gold_L": L_GOLD, "H": H.shape}

{'golay_L': 131072, 'mls_L': 131071, 'gold_L': 65535, 'H': (2, 2)}

## 3. Sygnały synchronizujące

In [75]:
def linear_chirp(fs, dur_s, f0=600.0, f1=6000.0, amp=0.6):
    n = int(round(dur_s * fs))
    t = np.arange(n, dtype=float) / fs
    k = (f1 - f0) / max(dur_s, 1e-12)
    sig = amp * np.sin(2.0 * np.pi * (f0 * t + 0.5 * k * t * t))
    fade = int(min(n // 10, fs * 0.005))
    ramp = np.linspace(0.0, 1.0, fade)
    sig[:fade] *= ramp
    sig[-fade:] *= ramp[::-1]
    return sig


def sine_tone(fs, dur_s, freq, amp=0.8):
    n = int(round(dur_s * fs))
    t = np.arange(n, dtype=float) / fs
    sig = amp * np.sin(2.0 * np.pi * freq * t)
    fade = int(min(n // 20, fs * 0.01))
    ramp = np.linspace(0.0, 1.0, fade)
    sig[:fade] *= ramp
    sig[-fade:] *= ramp[::-1]
    return sig


sync = linear_chirp(FS, SYNC_MS / 1000)
{"chirp_probek": sync.size}

{'chirp_probek': 1920}

In [76]:
channels = [[] for _ in range(N_SPK)]
manifest = []
cursor = [0]
guard = np.zeros(GUARD)
gap = np.zeros(int(round(GAP_MS / 1000 * FS)))


def append_all(name, kind, spk_data):
    max_len = max(len(d) for d in spk_data.values())
    for j in range(N_SPK):
        seg = np.asarray(spk_data[j], dtype=float) if j in spk_data else np.zeros(max_len)
        if seg.size < max_len:
            seg = np.pad(seg, (0, max_len - seg.size))
        channels[j].append(seg)
    manifest.append({"name": name, "kind": kind, "start": cursor[0],
                     "end": cursor[0] + max_len, "fs": FS})
    cursor[0] += max_len


def append_silence(name, length):
    append_all(name, "silence", {j: np.zeros(length) for j in range(N_SPK)})


def append_sync(label):
    append_all(f"sync_{label}", "sync", {j: sync for j in range(N_SPK)})
    append_silence(f"gap_after_sync_{label}", gap.size)


{"segmentow": len(manifest)}
append_sync("noise_floor")
append_silence("noise_floor", int(round(NOISE_FLOOR_MS / 1000 * FS)))
append_silence("gap_after_noise_floor", gap.size)

{"segmentow": len(manifest), "s": round(cursor[0] / FS, 1)}

{'segmentow': 4, 's': 5.4}

## 4. Solo Golay

In [77]:
append_sync("solo_golay")
for j in range(N_SPK):
    append_all(f"solo_golay_spk{j+1}_A", "solo_golay_A", {j: np.concatenate([A, guard])})
    append_all(f"solo_golay_spk{j+1}_B", "solo_golay_B", {j: np.concatenate([B, guard])})
append_silence("gap_after_solo_golay", gap.size)

{"segmentow": len(manifest), "s": round(cursor[0] / FS, 1)}

{'segmentow': 11, 's': 24.8}

## 5. Solo MLS

In [78]:
append_sync("solo_mls")
for j in range(N_SPK):
    append_all(f"solo_mls_spk{j+1}", "solo_mls", {j: np.concatenate([mls, guard])})
append_silence("gap_after_solo_mls", gap.size)

{"segmentow": len(manifest), "s": round(cursor[0] / FS, 1)}

{'segmentow': 16, 's': 34.7}

## 6. Golay + Hadamard

In [79]:
append_sync("golay_hadamard")
for i in range(H_SIZE):
    append_all(f"golay_had_row{i+1}_A", "golay_hadamard_A",
               {j: np.concatenate([H[i, j] * A, guard]) for j in range(N_SPK)})
    append_all(f"golay_had_row{i+1}_B", "golay_hadamard_B",
               {j: np.concatenate([H[i, j] * B, guard]) for j in range(N_SPK)})
append_silence("gap_after_golay_had", gap.size)

{"segmentow": len(manifest), "s": round(cursor[0] / FS, 1)}

{'segmentow': 23, 's': 54.1}

## 7. MLS + Hadamard

In [80]:
append_sync("mls_hadamard")
for i in range(H_SIZE):
    append_all(f"mls_had_row{i+1}", "mls_hadamard",
               {j: np.concatenate([H[i, j] * mls, guard]) for j in range(N_SPK)})
append_silence("gap_after_mls_had", gap.size)

{"segmentow": len(manifest), "s": round(cursor[0] / FS, 1)}

{'segmentow': 28, 's': 64.0}

## 8. MLS reciprocal i Gold

In [81]:
if N_SPK == 2:
    append_sync("mls_reciprocal")
    append_all("mls_reciprocal_pair", "mls_reciprocal",
               {0: np.concatenate([mls, guard]), 1: np.concatenate([mls_rec, guard])})
    append_silence("gap_after_mls_reciprocal", gap.size)

append_sync("gold")
append_all("gold_all", "gold",
           {j: np.concatenate([gold_codes[j], guard]) for j in range(N_SPK)})
append_silence("gap_after_gold", gap.size)

{"segmentow": len(manifest), "s": round(cursor[0] / FS, 1)}

{'segmentow': 36, 's': 72.9}

## 9. Sinusy THD

In [82]:
append_sync("thd")
for j in range(N_SPK):
    for freq in THD_FREQS:
        tone = sine_tone(FS, THD_DUR_MS / 1000.0, freq)
        append_all(f"thd_spk{j+1}_{freq}Hz", "thd", {j: np.concatenate([tone, guard])})
append_silence("gap_after_thd", gap.size)

{"segmentow": len(manifest), "s": round(cursor[0] / FS, 1)}

{'segmentow': 53, 's': 129.4}

## 10. Struktura

In [83]:
pd.DataFrame(sorted(Counter(s["kind"] for s in manifest).items()),
             columns=["rodzaj", "segmentow"])

,rodzaj,segmentow
0,golay_hadamard_A,2
1,golay_hadamard_B,2
2,gold,1
3,mls_hadamard,2
4,mls_reciprocal,1
5,silence,17
6,solo_golay_A,2
7,solo_golay_B,2
8,solo_mls,2
9,sync,8


## 11. Zapis

In [84]:
os.makedirs(OUT_DIR, exist_ok=True)
final = [np.concatenate(ch) for ch in channels]

data = np.column_stack([np.asarray(ch, dtype=np.float32) for ch in final])
mx = float(np.max(np.abs(data)))
if mx > 0:
    data *= PEAK / mx
sf.write(os.path.join(OUT_DIR, "bundle.wav"), data, FS, subtype="FLOAT")

meta = {"speakers": N_SPK, "fs": FS, "golay_n": GOLAY_N, "golay_L": L_GOLAY,
        "mls_m": MLS_M, "mls_L": L_MLS, "mls_reciprocal_L": L_MLS,
        "gold_m": GOLD_M, "gold_L": L_GOLD, "hadamard_size": H_SIZE,
        "guard": GUARD, "segments": manifest}
with open(os.path.join(OUT_DIR, "manifest.json"), "w", encoding="utf-8") as fh:
    json.dump(meta, fh, ensure_ascii=False, indent=2)

{p: os.path.getsize(os.path.join(OUT_DIR, p)) for p in sorted(os.listdir(OUT_DIR))}

{'bundle.wav': 49683496, 'manifest.json': 7820}